# Qwen-Image-2.1 UNCENSORED di Colab (via ComfyUI)

Model: `abenzerps/Qwen-Image-2.1-Uncensored-Quantized` — GGUF Q4_K_M, **tanpa safety checker / content filter**.

Cara kerja notebook ini: install ComfyUI + node GGUF → download 3 file model (~15 GB) → jalankan ComfyUI sebagai server lokal → generate lewat API.

## Checklist
1. Runtime → Change runtime type → **GPU** (T4 16GB cukup: diffusion 4,6 GB di VRAM, text encoder di RAM).
2. Run all, tunggu tiap cell selesai. Download model agak lama (sekali saja per sesi).
3. Generate pertama lambat (load model); generate berikutnya jauh lebih cepat.

> Notebook ini dibuat + workflow-nya **divalidasi via API** sebelum diserahkan. Kalau tetap ada error, screenshot-kan pesannya.

In [ ]:
# 1. Install ComfyUI + GGUF node (leejet fork = yang support Qwen-Image 2.1)
!git clone -q https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
!git clone -q https://github.com/leejet/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
!pip install -q -r /content/ComfyUI/requirements.txt gguf huggingface_hub
print("install OK")

In [ ]:
# 2. Download model (~15 GB total, sekali per sesi)
from huggingface_hub import hf_hub_download

REPO = "abenzerps/Qwen-Image-2.1-Uncensored-Quantized"
hf_hub_download(REPO, filename="qwen-image-2.1-UC-Q4_K_M.gguf",
               local_dir="/content/ComfyUI/models/diffusion_models")
hf_hub_download(REPO, filename="text_encoders/qwen3vl_8b_int8_convrot.safetensors",
               local_dir="/content/ComfyUI/models")
hf_hub_download(REPO, filename="vae/qwen_image_2.1_vae_bf16.safetensors",
               local_dir="/content/ComfyUI/models")
print("models OK")

In [ ]:
# 3. Start ComfyUI server di background
import subprocess, time, socket

subprocess.run(["pkill", "-f", "main.py.*8188"], capture_output=True); time.sleep(1)
log = open('/content/comfyui.log', 'w')
proc = subprocess.Popen(
    ["python", "/content/ComfyUI/main.py", "--port", "8188", "--listen", "127.0.0.1"],
    stdout=log, stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        socket.create_connection(("127.0.0.1", 8188), timeout=2).close()
        print("ComfyUI READY"); break
    except OSError:
        time.sleep(2)
else:
    print("TIMEOUT - cek /content/comfyui.log")

In [ ]:
# 4. Helper generate() — tidak perlu diubah
import json, time, uuid, random, urllib.request
from IPython.display import Image, display

SERVER = "127.0.0.1:8188"

def build_workflow(prompt, seed, width=1024, height=1024, steps=30):
    return {
        "1": {"class_type": "UnetLoaderGGUF",
              "inputs": {"unet_name": "qwen-image-2.1-UC-Q4_K_M.gguf"}},
        "2": {"class_type": "CLIPLoader",
              "inputs": {"clip_name": "qwen3vl_8b_int8_convrot.safetensors",
                          "type": "qwen_image"}},
        "3": {"class_type": "VAELoader",
              "inputs": {"vae_name": "qwen_image_2.1_vae_bf16.safetensors"}},
        "4": {"class_type": "CLIPTextEncode",
              "inputs": {"text": prompt, "clip": ["2", 0]}},
        "5": {"class_type": "CLIPTextEncode",
              "inputs": {"text": "", "clip": ["2", 0]}},
        "6": {"class_type": "EmptyLatentImage",
              "inputs": {"width": width, "height": height, "batch_size": 1}},
        "7": {"class_type": "KSampler",
              "inputs": {"model": ["1", 0], "positive": ["4", 0],
                          "negative": ["5", 0], "latent_image": ["6", 0],
                          "seed": seed, "steps": steps, "cfg": 2.5,
                          "sampler_name": "euler", "scheduler": "simple",
                          "denoise": 1.0}},
        "8": {"class_type": "VAEDecode",
              "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
        "9": {"class_type": "SaveImage",
              "inputs": {"images": ["8", 0], "filename_prefix": "qwen-uc"}},
    }

def generate(prompt, seed=None, width=1024, height=1024, steps=30):
    seed = seed if seed is not None else random.randint(0, 2**32 - 1)
    wf = build_workflow(prompt, seed, width, height, steps)
    req = urllib.request.Request(
        f"http://{SERVER}/prompt",
        data=json.dumps({"prompt": wf, "client_id": str(uuid.uuid4())}).encode(),
        headers={"Content-Type": "application/json"})
    prompt_id = json.load(urllib.request.urlopen(req))["prompt_id"]
    print(f"queued {prompt_id} | seed {seed} | {width}x{height} | {steps} steps")
    for _ in range(240):
        time.sleep(5)
        try:
            hist = json.load(urllib.request.urlopen(f"http://{SERVER}/history/{prompt_id}"))
        except Exception:
            continue
        if prompt_id in hist:
            break
    else:
        raise TimeoutError("render timeout — cek /content/comfyui.log")
    outs = hist[prompt_id]["outputs"]
    for node_out in outs.values():
        for img in node_out.get("images", []):
            fname = img["filename"]
            url = (f"http://{SERVER}/view?filename={fname}"
                   f"&subfolder={img.get('subfolder', '')}&type={img.get('type', 'output')}")
            out_path = f"/content/{fname}"
            urllib.request.urlretrieve(url, out_path)
            print("saved:", out_path)
            display(Image(out_path))
            return out_path
    raise RuntimeError("tidak ada gambar di output — cek /content/comfyui.log")

print("helper OK — pakai generate(\"prompt kamu\")")

In [ ]:
# 5. Contoh pertama
generate("a portrait of a cyberpunk warrior, neon city at night, highly detailed", seed=7)

## Bebas — ganti prompt sesukamu

Tips: `steps` 25–35 sweet spot (lebih tinggi = lebih detail, lebih lambat). `cfg` 2–4. Resolusi lain mis. `width=768, height=1024` buat portrait.